# **Block 1 — Imports**

We import the tools needed to load data, split train/test, train a decision tree, and evaluate it. Notice that we do not need scaling here, because decision trees do not rely on distances the way kNN does, so we can keep the setup even cleaner.

In [1]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score


# **Block 2 — Inputs (X = features, y = labels)**

A decision tree learns by asking a sequence of feature-based questions, like “Is this feature above a certain value?” so we still start the same way: inputs X and labels y. We use a built-in binary dataset so the focus stays on the tree concepts: rules, depth, and generalization.

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])

unique, counts = np.unique(y, return_counts=True)
print("Class counts:", dict(zip(unique, counts)))


X shape: (569, 30)
y shape: (569,)
First 10 labels: [0 0 0 0 0 0 0 0 0 0]
Class counts: {np.int64(0): np.int64(212), np.int64(1): np.int64(357)}


# **Block 3 — Train/Test split**

Even though trees can fit training data extremely well, the real question is whether those learned rules generalize. That is why we split into train and test and keep the test set untouched until evaluation, so we can see whether the tree learned signal or memorized noise.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train rows:", X_train.shape[0])
print("Test rows :", X_test.shape[0])


Train rows: 426
Test rows : 143


# **Block 4 — Train a baseline tree**

This block trains a basic decision tree with default settings so you can see the “raw behavior” of a tree. A key point is that trees are powerful and can become very complex very quickly, which is why we often compare a default tree to a depth-limited tree later.

In [4]:
# Baseline tree (defaults)
# By default, a tree can grow deep enough to fit training data very closely.
tree = DecisionTreeClassifier(random_state=42)

tree.fit(X_train, y_train)

print("Baseline decision tree trained.")


Baseline decision tree trained.


# **Block 5 — Predict + Evaluate (confusion matrix + metrics)**

This block evaluates the baseline tree on the test set. The confusion matrix shows what kinds of mistakes are happening, and the metrics summarize performance. The important habit is that evaluation happens on test data, not training data, because training accuracy can look “too good” with trees.

In [5]:
y_pred = tree.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred)

print("Accuracy :", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall   :", round(rec, 3))


Confusion matrix:
 [[49  4]
 [ 7 83]]
Accuracy : 0.923
Precision: 0.954
Recall   : 0.922


# **Block 6 — Compare train vs test accuracy (spot overfitting patterns)**

This block is here because trees can easily overfit, and the simplest way to detect that pattern is to compare training performance to testing performance. If training accuracy is much higher than test accuracy, the tree may have memorized details that do not generalize.

In [6]:
# Training accuracy (how well the tree fits what it has already seen)
train_pred = tree.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)

# Test accuracy (generalization)
test_acc = accuracy_score(y_test, y_pred)

print("Train accuracy:", round(train_acc, 3))
print("Test accuracy :", round(test_acc, 3))


Train accuracy: 1.0
Test accuracy : 0.923


# **Block 7 — Depth = complexity (shallow vs deeper tree)**

Depth is one of the cleanest ways to control tree complexity. A shallow tree asks fewer questions and is simpler, which can help prevent overfitting but can also underfit if it becomes too simple. This block compares a shallow tree to the baseline tree so you can see the tradeoff without adding lots of extra code.

In [7]:
# A depth-limited tree (simpler model)
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_tree.fit(X_train, y_train)

# Evaluate shallow tree on test data
shallow_pred = shallow_tree.predict(X_test)
shallow_test_acc = accuracy_score(y_test, shallow_pred)

# Compare baseline vs shallow
print("Baseline test accuracy:", round(test_acc, 3))
print("Shallow (max_depth=3) test accuracy:", round(shallow_test_acc, 3))


Baseline test accuracy: 0.923
Shallow (max_depth=3) test accuracy: 0.944
